# 05. Análisis complementario mediante clustering jerárquico (Ward)

Este cuaderno evalúa los agrupamientos obtenidos mediante el método de Ward para valores de k entre 2 y 8, utilizando los cinco indicadores de la configuración E1 y las 5 626 subpartidas de la base analítica.

Se emplea la distancia euclidiana y se estandarizan los indicadores mediante StandardScaler. La solución de tres grupos se compara con la obtenida mediante K-means.

## Archivo de entrada

- `resultados/clustering/subpartidas_E1_k3_FINAL.csv`

## Archivos de salida

Los resultados se almacenan en `resultados/clustering/ward_complementario/`.

El análisis es complementario y no modifica la clasificación definitiva obtenida mediante K-means.

## 1. Carga y validación de la base final

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score
)
from IPython.display import display

# Rutas del archivo de entrada.
rutas_posibles = [
    Path("resultados/clustering/subpartidas_E1_k3_FINAL.csv"),
    Path("subpartidas_E1_k3_FINAL.csv"),
]
ruta_csv = next((p for p in rutas_posibles if p.is_file()), None)
if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró el archivo subpartidas_E1_k3_FINAL.csv "
        "en las rutas especificadas."
    )

variables_E1 = [
    "log_fob_total",
    "anios_activos",
    "episodios_exportadores",
    "regularidad_mensual",
    "hhi_destinos_periodo",
]

base = pd.read_csv(ruta_csv, dtype={"subpartida_homologada": "string"})
columnas = ["subpartida_homologada", "grupo_E1"] + variables_E1
faltantes = sorted(set(columnas) - set(base.columns))
if faltantes:
    raise ValueError(f"Faltan columnas en el CSV: {faltantes}")

assert len(base) == 5626, f"Se esperaban 5626 filas, hay {len(base)}"
assert base["subpartida_homologada"].nunique() == len(base), "Hay subpartidas repetidas"
assert base[columnas].notna().all().all(), "Hay valores faltantes"
assert np.isfinite(base[variables_E1].to_numpy(dtype=float)).all(), "Hay valores infinitos"
base["grupo_E1"] = base["grupo_E1"].astype(int)
assert base["grupo_E1"].value_counts().sort_index().to_dict() == {
    1: 2692, 2: 1780, 3: 1154
}, "Los grupos de K-means no coinciden con la exportación final"

# Estandarización de los indicadores de la configuración E1.
X = StandardScaler().fit_transform(base[variables_E1])
etiquetas_kmeans = base["grupo_E1"].to_numpy()

salida = Path("resultados/clustering/ward_complementario")
salida.mkdir(parents=True, exist_ok=True)

print("Archivo:", ruta_csv)
print("Subpartidas:", len(base))
print("Variables E1:", variables_E1)
print("Grupos definitivos K-means:", base["grupo_E1"].value_counts().sort_index().to_dict())

## 2. Construcción de la jerarquía de Ward y cortes k = 2 a 8

Se calcula **una sola jerarquía** y después se extraen los agrupamientos correspondientes a cada número de grupos. En Ward los conjuntos son anidados; los números de grupo no tienen significado comercial previo.

In [ ]:
Z_ward = linkage(X, method="ward", metric="euclidean")
valores_k = list(range(2, 9))
etiquetas_ward = {
    k: fcluster(Z_ward, t=k, criterion="maxclust") for k in valores_k
}

conteos = []
for k, etiq in etiquetas_ward.items():
    valores, cantidades = np.unique(etiq, return_counts=True)
    assert len(valores) == k, f"Ward devolvió {len(valores)} grupos para k={k}"
    conteos.append({"k": k, **{f"grupo_{g}": int(n) for g, n in zip(valores, cantidades)}})

tamanos = pd.DataFrame(conteos).set_index("k")
print("Tamaños de grupos Ward (k = 2 a 8):")
display(tamanos.fillna(0).astype(int))
tamanos.to_csv(salida / "ward_tamanos_k2_k8.csv", encoding="utf-8-sig")

# Exportar etiquetas por subpartida para auditoría posterior.
asignaciones = base[["subpartida_homologada", "grupo_E1"]].copy()
for k in valores_k:
    asignaciones[f"Ward_k{k}"] = etiquetas_ward[k]
asignaciones.to_csv(salida / "ward_asignaciones_k2_k8.csv", index=False, encoding="utf-8-sig")

## 3. Validación interna para k = 2 a 8

Se calculan los índices de silueta, Davies-Bouldin y Calinski-Harabasz sobre la matriz estandarizada de la configuración E1.

La evaluación se realiza considerando las 5 626 subpartidas.

In [ ]:
muestra_silhouette = None   # Evaluación con todas las observaciones.
filas = []
for k in valores_k:
    etiquetas = etiquetas_ward[k]
    print(f"Evaluando Ward k={k} ...")
    sil = silhouette_score(
        X, etiquetas,
        sample_size=muestra_silhouette,
        random_state=42
    )
    filas.append({
        "metodo": "Ward", "k": k,
        "Silhouette": sil,
        "Davies-Bouldin": davies_bouldin_score(X, etiquetas),
        "Calinski-Harabasz": calinski_harabasz_score(X, etiquetas)
    })

metricas_ward = pd.DataFrame(filas)
print("\nMÉTRICAS DE WARD, k = 2 a 8")
display(metricas_ward.round(4))
metricas_ward.to_csv(salida / "ward_metricas_k2_k8.csv", index=False, encoding="utf-8-sig")

print("\nSolución con mejor valor de cada métrica en el intervalo analizado:")
for nombre, optimizacion in [
    ("Silhouette", "max"),
    ("Davies-Bouldin", "min"),
    ("Calinski-Harabasz", "max"),
]:
    indice = (metricas_ward[nombre].idxmax() if optimizacion == "max"
              else metricas_ward[nombre].idxmin())
    print(f"{nombre}: k={int(metricas_ward.loc[indice, 'k'])}, "
          f"valor={metricas_ward.loc[indice, nombre]:.4f}")

if muestra_silhouette is not None:
    print("El índice de Silhouette se calculó sobre una muestra.")

## 4. Representación gráfica de las métricas de Ward

La línea vertical señala la solución de tres grupos utilizada para la comparación con K-means.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for ax, nombre, sentido in zip(
    ejes,
    ["Silhouette", "Davies-Bouldin", "Calinski-Harabasz"],
    ["Mayor es mejor", "Menor es mejor", "Mayor es mejor"]
):
    ax.plot(metricas_ward["k"], metricas_ward[nombre], marker="o")
    ax.axvline(3, linestyle="--", linewidth=1, alpha=0.6)
    ax.set_xticks(valores_k)
    ax.set_xlabel("Número de grupos (k)")
    ax.set_ylabel(nombre)
    ax.set_title(f"{nombre}\n{sentido}")
    ax.grid(alpha=0.2)
fig.suptitle("Ward · Indicadores E1 · Evaluación de k = 2 a 8", fontweight="bold")
fig.tight_layout()
fig.savefig(salida / "ward_metricas_k2_k8.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Correspondencia entre los agrupamientos de Ward con k = 2 y k = 3

In [ ]:
cruce_2_3 = pd.crosstab(
    pd.Series(etiquetas_ward[2], name="Ward k=2"),
    pd.Series(etiquetas_ward[3], name="Ward k=3")
)
print("Correspondencia dentro de la jerarquía de Ward:")
display(cruce_2_3)
cruce_2_3.to_csv(salida / "ward_transicion_k2_k3.csv", encoding="utf-8-sig")

## 6. Caracterización de los grupos de Ward

Se calculan los promedios de los indicadores antes de la estandarización, con el propósito de caracterizar los grupos obtenidos mediante el método de Ward.

In [ ]:
perfiles_por_k = {}
for k in valores_k:
    temp = base[variables_E1].copy()
    temp["grupo_Ward"] = etiquetas_ward[k]
    p = temp.groupby("grupo_Ward")[variables_E1].mean()
    p.insert(0, "subpartidas", temp.groupby("grupo_Ward").size())
    perfiles_por_k[k] = p
    p.to_csv(salida / f"ward_perfiles_k{k}.csv", encoding="utf-8-sig")

print("Perfiles Ward con k=2:")
display(perfiles_por_k[2].round(3))
print("Perfiles Ward con k=3:")
display(perfiles_por_k[3].round(3))
print("Perfiles Ward con k=4:")
display(perfiles_por_k[4].round(3))

## 7. Dendrograma con cortes para k = 2, k = 3 y k = 4

El dendrograma representa la estructura jerárquica de los agrupamientos y las distancias de fusión. Se presenta una versión abreviada para facilitar su visualización.

In [ ]:
h_3 = (Z_ward[-3, 2] + Z_ward[-2, 2]) / 2
h_2 = (Z_ward[-2, 2] + Z_ward[-1, 2]) / 2
h_4 = (Z_ward[-4, 2] + Z_ward[-3, 2]) / 2

fig, ax = plt.subplots(figsize=(12, 5.8))
dendrogram(
    Z_ward, truncate_mode="lastp", p=18,
    show_leaf_counts=True, color_threshold=0,
    above_threshold_color="#587187", ax=ax
)
ax.axhline(h_2, linestyle="--", linewidth=1.7, label="Corte k = 2")
ax.axhline(h_3, linestyle=":", linewidth=2, label="Corte k = 3")
ax.axhline(h_4, linestyle=":", linewidth=1.7,color="#A05C87", label="Corte k = 4")
ax.set_title("Dendrograma de la configuración E1 mediante el método de Ward", fontweight="bold")
ax.set_xlabel("Agrupaciones finales (cantidad de subpartidas entre paréntesis)")
ax.set_ylabel("Distancia de fusión de Ward")
ax.legend(loc="upper left", frameon=False)
ax.grid(axis="y", alpha=0.16)
fig.tight_layout()
fig.savefig(salida / "ward_dendrograma_k2_k3_k4.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Comparación entre Ward y K-means con k = 3

Se comparan las asignaciones obtenidas mediante ambos métodos utilizando el índice de Rand ajustado, las tablas de correspondencia y las métricas de validación interna.

In [ ]:
lab_w3 = etiquetas_ward[3]
ari = adjusted_rand_score(etiquetas_kmeans, lab_w3)

cruce_metodos = pd.crosstab(
    pd.Series(etiquetas_kmeans, name="K-means E1 k=3"),
    pd.Series(lab_w3, name="Ward k=3")
)
porcentajes = cruce_metodos.div(cruce_metodos.sum(axis=1), axis=0) * 100
print(f"ARI entre Ward k=3 y K-means E1 k=3: {ari:.4f}")
print("Correspondencia (cantidades):")
display(cruce_metodos)
print("Porcentaje dentro de cada grupo K-means:")
display(porcentajes.round(1))
cruce_metodos.to_csv(salida / "ward_vs_kmeans_k3.csv", encoding="utf-8-sig")

# Cálculo de las métricas de K-means sobre la misma matriz estandarizada.
fila_kmeans = {
    "metodo": "K-means E1", "k": 3,
    "Silhouette": silhouette_score(
        X, etiquetas_kmeans,
        sample_size=muestra_silhouette,
        random_state=42
    ),
    "Davies-Bouldin": davies_bouldin_score(X, etiquetas_kmeans),
    "Calinski-Harabasz": calinski_harabasz_score(X, etiquetas_kmeans)
}
comparacion = pd.concat([
    metricas_ward.query("k == 3"),
    pd.DataFrame([fila_kmeans])
], ignore_index=True)
print("Comparación entre métodos con k=3:")
display(comparacion.round(4))
comparacion.to_csv(salida / "comparacion_ward_kmeans_k3.csv", index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(6.5, 4.9))
mat = porcentajes.to_numpy()
ax.imshow(mat, cmap="Blues", vmin=0, vmax=100)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        ax.text(j, i,
                f"{cruce_metodos.iat[i,j]:,}\n({mat[i,j]:.1f} %)",
                ha="center", va="center", fontsize=10,
                color="white" if mat[i,j] > 55 else "#203040")
ax.set_xticks(range(3), [f"Ward-G{x}" for x in cruce_metodos.columns])
ax.set_yticks(range(3), [f"E1-G{x}" for x in cruce_metodos.index])
ax.set_xlabel("Ward (k=3)")
ax.set_ylabel("K-means E1 (k=3)")
ax.set_title(f"Correspondencia entre métodos · ARI = {ari:.4f}")
fig.tight_layout()
fig.savefig(salida / "ward_vs_kmeans_k3.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Archivos generados

Los resultados se almacenan en `resultados/clustering/ward_complementario/` e incluyen:

- Métricas de validación interna.
- Tamaños y asignaciones de los grupos.
- Promedios de los indicadores por grupo.
- Correspondencias entre los agrupamientos.
- Comparación entre Ward y K-means.
- Representaciones gráficas.